# 02 — Canonical time-series EDA

**Purpose:** establish what the canonical telemetry contains, whether its
time structure supports anomaly-detection features, and which limitations
must be carried into modelling.

The notebook runs unchanged for Telecom and Petrobras 3W. It reads only
`SPEC-CORE`. It does not create anomaly scores, open labels, remove outliers,
tune thresholds or fit a production model.


## 1. Setup and analysis scope

Structural checks use the complete canonical dataset through DuckDB. More
expensive plots and time-series tests use a deterministic, label-blind
subset and the first 40% of each selected entity's observed history.

This interval is frozen evidence, not guaranteed normal data. The row budget
is checked before materialising the analysis frame.


In [ ]:
import hashlib
import os
import shutil
import subprocess
import sys
import tempfile
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from scipy.stats import skew
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller, kpss

try:
    import duckdb
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "duckdb"])
    import duckdb

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")

DRIVE_ROOT = Path(os.getenv(
    "ANOMALY_DRIVE_ROOT",
    "/content/drive/MyDrive/anomaly_detection",
))
NOTEBOOK_HOME = Path(os.getenv(
    "ANOMALY_NOTEBOOK_HOME",
    DRIVE_ROOT / "research" / "week1",
))
if str(NOTEBOOK_HOME) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_HOME))

from week1_core import CORE_VERSION, immutable_directory, read_json, sha256_file, write_json

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)

SECTOR = os.getenv("EDA_SECTOR", "telecom")  # or "petrobras_3w"
CANONICAL_RUN_IDS = {
    "telecom": "telecom_core_v0_6_run1",
    "petrobras_3w": "petrobras_3w_core_v0_6_run1",
}
if SECTOR not in CANONICAL_RUN_IDS:
    raise ValueError(f"Choose one of {list(CANONICAL_RUN_IDS)}")

CANONICAL_RUN_ID = os.getenv("EDA_CANONICAL_RUN_ID", CANONICAL_RUN_IDS[SECTOR])
EDA_RUN_ID = os.getenv("EDA_RUN_ID", f"{SECTOR}_eda_v0_2_run1")
CORE_ROOT = (
    DRIVE_ROOT / "outputs" / "canonical" / f"v{CORE_VERSION}"
    / SECTOR / CANONICAL_RUN_ID / "SPEC-CORE"
)
EDA_ROOT = DRIVE_ROOT / "outputs" / "eda" / "v0.2.0" / SECTOR / EDA_RUN_ID

CALIBRATION_FRACTION = 0.40
ENTITY_LIMIT = int(os.getenv("EDA_ENTITY_LIMIT", "20"))
MAX_ANALYSIS_ROWS = int(os.getenv("EDA_MAX_ANALYSIS_ROWS", "4000000"))
MAX_PLOT_METRICS = int(os.getenv("EDA_MAX_PLOT_METRICS", "8"))
MAX_PLOT_POINTS = 5_000
MIN_POPULATION_SERIES = 10
MIN_STL_CYCLES = 6
MAX_SHORT_GAP_FILL = 4
SAVE_OUTPUTS = os.getenv("SAVE_EDA_OUTPUTS", "1") == "1"

figure_temp = tempfile.TemporaryDirectory()
FIGURE_CACHE = Path(figure_temp.name)

def save_and_show(fig, filename):
    fig.tight_layout()
    fig.savefig(FIGURE_CACHE / filename, dpi=140, bbox_inches="tight")
    plt.show()
    plt.close(fig)

display(pd.Series({
    "sector": SECTOR,
    "spec_core": str(CORE_ROOT),
    "eda_output": str(EDA_ROOT),
    "entity_limit": ENTITY_LIMIT,
    "max_analysis_rows": MAX_ANALYSIS_ROWS,
}, name="value").to_frame())


## 2. File-access guard and contract inspection

Every permitted input is resolved below `SPEC-CORE`. Neither `SPEC-EVAL`
nor optional context is visible to this notebook.


In [ ]:
CORE_RESOLVED = CORE_ROOT.resolve()
if not CORE_RESOLVED.is_dir():
    raise FileNotFoundError(f"Run Notebook 01B first: {CORE_ROOT}")

def core_path(relative_path):
    path = (CORE_ROOT / relative_path).resolve()
    if not path.is_relative_to(CORE_RESOLVED):
        raise PermissionError(f"EDA read escaped SPEC-CORE: {path}")
    return path

manifest_path = core_path("manifest.json")
catalogue_path = core_path("metric_catalogue.parquet")
registry_path = core_path("entity_registry.parquet")
gaps_path = core_path("collection_gaps.parquet")
telemetry_parts = sorted(core_path("telemetry").glob("part-*.parquet"))
if not telemetry_parts:
    raise FileNotFoundError("No canonical telemetry parts found")

core_manifest = read_json(manifest_path)
catalogue = pd.read_parquet(catalogue_path)
registry = pd.read_parquet(registry_path)
collection_gaps = pd.read_parquet(gaps_path)
assert core_manifest["contract_version"] == CORE_VERSION

display(catalogue)
display(registry.head(10))
display(collection_gaps.head(10))
display(pd.Series(core_manifest["row_counts"], name="rows").to_frame())


## 3. Complete structural audit

DuckDB scans every Parquet part without concatenating the complete long table
in notebook memory. The duplicate check is global across partitions.

Two distinct quantities are reported:

- **value validity:** a row exists and its value is usable;
- **coverage:** expected periodic observations that actually have rows.

Coverage is intentionally unavailable for 3W event recordings because no
continuous reporting obligation exists between files.


In [ ]:
telemetry_glob = str(core_path("telemetry") / "*.parquet").replace("'", "''")
connection = duckdb.connect()
connection.execute(
    f"CREATE VIEW telemetry AS SELECT * FROM read_parquet('{telemetry_glob}')"
)

structural = connection.execute('''
    SELECT
        count(*) AS rows,
        count(DISTINCT entity_id) AS entities,
        count(DISTINCT metric_id) AS metrics,
        min(event_ts) AS first_ts,
        max(event_ts) AS last_ts,
        count(*) - count(DISTINCT (event_ts, entity_id, metric_id)) AS duplicate_keys
    FROM telemetry
''').df()
quality_summary = connection.execute('''
    SELECT metric_id, quality_code, count(*) AS rows
    FROM telemetry
    GROUP BY metric_id, quality_code
    ORDER BY metric_id, quality_code
''').df()
observed_series = connection.execute('''
    SELECT
        entity_id,
        metric_id,
        count(*) AS n_rows,
        count(value) AS n_non_null,
        sum(CASE WHEN quality_code = 'invalid' THEN 1 ELSE 0 END) AS n_invalid,
        sum(CASE WHEN quality_code = 'clipped' THEN 1 ELSE 0 END) AS n_clipped,
        min(event_ts) AS first_ts,
        max(event_ts) AS last_ts
    FROM telemetry
    GROUP BY entity_id, metric_id
''').df()

expected = registry[["entity_id", "observed_from", "observed_to"]].merge(
    catalogue[["metric_id", "sampling_mode", "expected_cadence_seconds"]],
    how="cross",
)
expected["observed_from"] = pd.to_datetime(expected["observed_from"], utc=True)
expected["observed_to"] = pd.to_datetime(expected["observed_to"], utc=True)
periodic = expected["sampling_mode"].eq("periodic") & expected["expected_cadence_seconds"].notna()
elapsed = (expected["observed_to"] - expected["observed_from"]).dt.total_seconds()
expected["expected_rows"] = np.where(
    periodic,
    np.floor(elapsed / expected["expected_cadence_seconds"]) + 1,
    np.nan,
)
expected["coverage_basis"] = np.where(periodic, "observed_bounds", "not_applicable_recordings")

full_series = observed_series.merge(
    expected[["entity_id", "metric_id", "expected_rows", "coverage_basis"]],
    on=["entity_id", "metric_id"],
    how="left",
    validate="one_to_one",
)
full_series["valid_rate"] = (full_series["n_rows"] - full_series["n_invalid"]) / full_series["n_rows"]
full_series["clipped_rate"] = full_series["n_clipped"] / full_series["n_rows"]
full_series["coverage"] = full_series["n_rows"] / full_series["expected_rows"]

display(structural)
display(quality_summary)
display(full_series.head(20))
assert int(structural.loc[0, "duplicate_keys"]) == 0, "Canonical telemetry keys are not globally unique"


## 4. View the canonical data

The first table shows the long contract. The second pivots a small sample so
the measurements can be read as an ordinary multivariate time series.


In [ ]:
sample_long = connection.execute('''
    SELECT * FROM telemetry
    ORDER BY entity_id, event_ts, metric_id
    LIMIT 200
''').df()
sample_wide = sample_long.pivot_table(
    index=["event_ts", "entity_id"],
    columns="metric_id",
    values="value",
    aggfunc="first",
).reset_index()
display(sample_long.head(20))
display(sample_wide.head(10))


## 5. Freeze the label-blind analysis window

Entities are selected by a stable hash, not by interesting behaviour. The
analysis frame is reduced deterministically if its estimated row count would
exceed the configured memory budget. The budget is checked before `.df()`.


In [ ]:
def stable_key(value):
    return hashlib.sha256(str(value).encode()).hexdigest()

candidates = sorted(registry["entity_id"].astype(str), key=stable_key)
selected_entities = candidates[:min(ENTITY_LIMIT, len(candidates))]

def make_windows(entity_ids):
    windows = registry.loc[registry["entity_id"].astype(str).isin(entity_ids)].copy()
    windows["observed_from"] = pd.to_datetime(windows["observed_from"], utc=True)
    windows["observed_to"] = pd.to_datetime(windows["observed_to"], utc=True)
    windows["analysis_start"] = windows["observed_from"]
    windows["analysis_end"] = windows["observed_from"] + (
        windows["observed_to"] - windows["observed_from"]
    ) * CALIBRATION_FRACTION
    return windows[["entity_id", "analysis_start", "analysis_end"]]

while True:
    analysis_windows = make_windows(selected_entities)
    connection.register("analysis_windows", analysis_windows)
    estimated_rows = connection.execute('''
        SELECT count(*)
        FROM telemetry AS t
        JOIN analysis_windows AS w USING (entity_id)
        WHERE t.event_ts BETWEEN w.analysis_start AND w.analysis_end
    ''').fetchone()[0]
    if estimated_rows <= MAX_ANALYSIS_ROWS or len(selected_entities) <= 4:
        break
    selected_entities = selected_entities[:-1]

if estimated_rows > MAX_ANALYSIS_ROWS:
    raise MemoryError(
        f"Analysis requires {estimated_rows:,} rows; increase EDA_MAX_ANALYSIS_ROWS "
        "or lower EDA_ENTITY_LIMIT."
    )

calibration = connection.execute('''
    SELECT t.*
    FROM telemetry AS t
    JOIN analysis_windows AS w USING (entity_id)
    WHERE t.event_ts BETWEEN w.analysis_start AND w.analysis_end
    ORDER BY t.entity_id, t.metric_id, t.event_ts
''').df()
calibration["event_ts"] = pd.to_datetime(calibration["event_ts"], utc=True)
calibration["value"] = pd.to_numeric(calibration["value"], errors="coerce")

display(analysis_windows)
print(f"Selected entities: {len(selected_entities)}")
print(f"Materialised analysis rows: {len(calibration):,}")


## 6. Per-series descriptive statistics

Every difference and transition statistic is calculated inside one
`(entity_id, metric_id)` series. Metric-level summaries are then medians and
quantiles across series; entities are never pooled before differencing.


In [ ]:
def mad(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if not len(values):
        return np.nan
    center = np.median(values)
    return np.median(np.abs(values - center))


def level_scale_correlation(values, blocks=12):
    chunks = [chunk for chunk in np.array_split(values, blocks) if len(chunk) >= 5]
    if len(chunks) < 3:
        return np.nan
    levels = np.array([np.median(chunk) for chunk in chunks])
    scales = np.array([mad(chunk) for chunk in chunks])
    return pd.Series(levels).corr(pd.Series(scales), method="spearman")


def describe_series(group):
    assert group["entity_id"].nunique() == 1
    assert group["metric_id"].nunique() == 1
    ordered = group.sort_values("event_ts")
    values = ordered.loc[ordered["quality_code"].ne("invalid"), "value"].dropna().to_numpy()
    differences = np.diff(values) if len(values) > 1 else np.array([])
    first, second = np.array_split(values, 2) if len(values) else ([], [])
    scale = mad(values)
    return {
        "n_rows_analysis": len(ordered),
        "n_valid_analysis": len(values),
        "valid_rate_analysis": len(values) / len(ordered) if len(ordered) else np.nan,
        "clipped_rate_analysis": ordered["quality_code"].eq("clipped").mean(),
        "unique_values": pd.Series(values).nunique(),
        "minimum": np.min(values) if len(values) else np.nan,
        "q05": np.quantile(values, 0.05) if len(values) else np.nan,
        "median": np.median(values) if len(values) else np.nan,
        "q95": np.quantile(values, 0.95) if len(values) else np.nan,
        "maximum": np.max(values) if len(values) else np.nan,
        "mad": scale,
        "skewness": skew(values, bias=False) if len(values) >= 3 and np.unique(values).size > 2 else np.nan,
        "zero_rate": np.mean(values == 0) if len(values) else np.nan,
        "negative_difference_rate": np.mean(differences < 0) if len(differences) else np.nan,
        "transition_rate": np.mean(differences != 0) if len(differences) else np.nan,
        "drift_effect_mad": (
            (np.median(second) - np.median(first)) / scale
            if len(first) and len(second) and scale > 0 else np.nan
        ),
        "level_scale_correlation": level_scale_correlation(values),
    }


series_rows = []
for (entity_id, metric_id), group in calibration.groupby(["entity_id", "metric_id"], sort=True):
    series_rows.append({
        "entity_id": str(entity_id),
        "metric_id": str(metric_id),
        **describe_series(group),
    })
series_profile = pd.DataFrame(series_rows).merge(
    full_series[[
        "entity_id", "metric_id", "n_rows", "valid_rate",
        "clipped_rate", "coverage", "coverage_basis",
    ]].rename(columns={
        "n_rows": "n_rows_full",
        "valid_rate": "valid_rate_full",
        "clipped_rate": "clipped_rate_full",
    }),
    on=["entity_id", "metric_id"],
    how="left",
    validate="one_to_one",
)

def series_status(row):
    if row["n_valid_analysis"] < 50 or row["unique_values"] <= 1:
        return "insufficient_values"
    if row["valid_rate_full"] < 0.80:
        return "limited_value_validity"
    if pd.notna(row["coverage"]) and row["coverage"] < 0.80:
        return "limited_coverage"
    if pd.isna(row["coverage"]):
        return "coverage_not_applicable"
    return "descriptively_usable"

series_profile["series_status"] = series_profile.apply(series_status, axis=1)
display(series_profile.head(30))


In [ ]:
RATE_COLUMNS = [
    "valid_rate_full", "clipped_rate_full", "coverage",
    "valid_rate_analysis", "zero_rate",
]
TEMPORAL_COLUMNS = [
    "negative_difference_rate", "transition_rate",
    "drift_effect_mad", "level_scale_correlation", "skewness",
]

metric_rows = []
for metric_id, group in series_profile.groupby("metric_id", sort=True):
    row = {
        "metric_id": metric_id,
        "series_count": len(group),
        "between_entity_mad": mad(group["median"]),
        "median_within_entity_mad": group["mad"].median(),
    }
    for column in RATE_COLUMNS:
        row[f"{column}_p10"] = group[column].quantile(0.10)
        row[f"{column}_p50"] = group[column].median()
        row[f"{column}_p90"] = group[column].quantile(0.90)
    for column in TEMPORAL_COLUMNS:
        row[f"{column}_p50"] = group[column].median()
        row[f"{column}_p90"] = group[column].quantile(0.90)
        row[f"{column}_series_n"] = group[column].notna().sum()
    metric_rows.append(row)

metric_profile = pd.DataFrame(metric_rows).merge(
    catalogue,
    on="metric_id",
    how="left",
    validate="one_to_one",
)

status_counts = (
    series_profile.groupby(["metric_id", "series_status"]).size()
    .unstack(fill_value=0)
    .reset_index()
)
metric_profile = metric_profile.merge(status_counts, on="metric_id", how="left")
display(metric_profile)


## 7. Missingness, censoring and distributions

Rates are shown across entities rather than as one pooled percentage. For
periodic telemetry the heatmap displays coverage. For event recordings,
where coverage between files is undefined, it displays value validity.


In [ ]:
heatmap_value = "coverage" if series_profile["coverage"].notna().any() else "valid_rate_full"
heatmap = series_profile.pivot(index="entity_id", columns="metric_id", values=heatmap_value)
fig, ax = plt.subplots(figsize=(max(12, len(heatmap.columns) * 0.55), max(5, len(heatmap) * 0.28)))
sns.heatmap(heatmap, vmin=0, vmax=1, cmap="viridis", ax=ax)
ax.set_title(f"Per-series {heatmap_value.replace('_', ' ')}")
save_and_show(fig, "series_coverage_or_validity.png")

plot_metrics = []
for kind in catalogue["measurement_kind"].drop_duplicates():
    candidates = metric_profile.loc[metric_profile["measurement_kind"].eq(kind), "metric_id"]
    if len(candidates):
        plot_metrics.append(candidates.iloc[0])
for metric_id in metric_profile["metric_id"]:
    if metric_id not in plot_metrics:
        plot_metrics.append(metric_id)
    if len(plot_metrics) >= MAX_PLOT_METRICS:
        break

fig, axes = plt.subplots(len(plot_metrics), 2, figsize=(14, 4 * len(plot_metrics)), squeeze=False)
for row, metric_id in enumerate(plot_metrics):
    values = calibration.loc[
        calibration["metric_id"].eq(metric_id) & calibration["quality_code"].ne("invalid"),
        "value",
    ].dropna()
    if len(values) > 50_000:
        values = values.sample(50_000, random_state=42)
    if values.empty:
        axes[row, 0].set_title(f"{metric_id}: no valid values")
        axes[row, 1].axis("off")
        continue
    lower, upper = values.quantile([0.005, 0.995])
    sns.histplot(values.loc[values.between(lower, upper)], bins=40, ax=axes[row, 0])
    sns.boxplot(x=values, showfliers=False, ax=axes[row, 1])
    axes[row, 0].set_title(f"{metric_id}: central 99%")
    axes[row, 1].set_title(f"{metric_id}: robust boxplot")
save_and_show(fig, "distributions.png")


## 8. Representative temporal behaviour

One deterministic representative series is used per metric. ACF uses pairs
exactly one cadence apart, so gaps do not compress time. ADF and KPSS use the
longest regular segment after filling only short internal gaps; the number
of observations used is always reported.


In [ ]:
representatives = (
    series_profile.sort_values(
        ["metric_id", "valid_rate_full", "n_valid_analysis", "entity_id"],
        ascending=[True, False, False, True],
    )
    .groupby("metric_id", as_index=False)
    .first()[["metric_id", "entity_id"]]
)
catalogue_index = catalogue.set_index("metric_id")

def raw_series(entity_id, metric_id):
    frame = calibration.loc[
        calibration["entity_id"].astype(str).eq(str(entity_id))
        & calibration["metric_id"].eq(metric_id)
        & calibration["quality_code"].ne("invalid"),
        ["event_ts", "value"],
    ].dropna().drop_duplicates("event_ts").sort_values("event_ts")
    return frame.set_index("event_ts")["value"]

def longest_segment(series, cadence):
    if series.empty:
        return series
    breaks = series.index.to_series().diff().gt(cadence * 1.5).cumsum()
    longest = breaks.value_counts().idxmax()
    return series.loc[breaks.eq(longest).to_numpy()]

def regular_segment(series, cadence, sampling_mode):
    if series.empty:
        return series, 0.0, np.nan
    source = longest_segment(series, cadence) if sampling_mode == "recording" else series
    regular = source.asfreq(cadence)
    filled = regular.interpolate(
        limit=MAX_SHORT_GAP_FILL,
        limit_area="inside",
    )
    fill_fraction = (filled.notna().sum() - regular.notna().sum()) / len(regular)
    regular_missing_rate = regular.isna().mean()
    valid = filled.notna()
    if not valid.any():
        return filled.iloc[:0], fill_fraction, regular_missing_rate
    runs = valid.ne(valid.shift()).cumsum()
    longest = runs.loc[valid].value_counts().idxmax()
    segment = filled.loc[runs.eq(longest)].dropna()
    return segment, fill_fraction, regular_missing_rate

def lag_one_correlation(series, cadence):
    current = series.rename("current").to_frame()
    previous = pd.Series(series.to_numpy(), index=series.index + cadence, name="previous")
    pairs = current.join(previous).dropna()
    return pairs["current"].corr(pairs["previous"]), len(pairs)

def stationarity_tests(values):
    values = pd.Series(values).dropna().iloc[:10_000]
    result = {
        "stationarity_n": len(values),
        "adf_pvalue": np.nan,
        "kpss_pvalue": np.nan,
        "stationarity_evidence": "insufficient_data",
    }
    if len(values) < 100 or values.nunique() < 3:
        return result
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            adf = adfuller(values, maxlag=min(12, len(values) // 10), autolag="AIC")
            kpss_result = kpss(values, regression="c", nlags="auto")
    except (ValueError, np.linalg.LinAlgError):
        result["stationarity_evidence"] = "test_failed"
        return result
    result["adf_pvalue"] = adf[1]
    result["kpss_pvalue"] = kpss_result[1]
    if adf[1] <= 0.05 and kpss_result[1] > 0.05:
        result["stationarity_evidence"] = "stationary_evidence"
    elif adf[1] > 0.05 and kpss_result[1] <= 0.05:
        result["stationarity_evidence"] = "nonstationary_evidence"
    else:
        result["stationarity_evidence"] = "mixed_evidence"
    return result


In [ ]:
temporal_rows = []
for row in representatives.itertuples(index=False):
    metadata = catalogue_index.loc[row.metric_id]
    series = raw_series(row.entity_id, row.metric_id)
    cadence_seconds = pd.to_numeric(metadata["expected_cadence_seconds"], errors="coerce")
    if pd.isna(cadence_seconds):
        recording = series
        regular = series.iloc[:0]
        fill_fraction, regular_missing_rate = np.nan, np.nan
        acf_1, acf_pairs = np.nan, 0
    else:
        cadence = pd.Timedelta(seconds=float(cadence_seconds))
        recording = longest_segment(series, cadence)
        regular, fill_fraction, regular_missing_rate = regular_segment(
            series, cadence, metadata["sampling_mode"]
        )
        acf_1, acf_pairs = lag_one_correlation(series, cadence)
    differences = recording.diff().dropna()
    temporal_rows.append({
        "metric_id": row.metric_id,
        "entity_id": row.entity_id,
        "sampling_mode": metadata["sampling_mode"],
        "expected_cadence_seconds": cadence_seconds,
        "raw_observations": len(series),
        "longest_recording_observations": len(recording),
        "regular_test_observations": len(regular),
        "short_gap_fill_fraction": fill_fraction,
        "regular_missing_rate": regular_missing_rate,
        "acf_1": acf_1,
        "acf_pairs": acf_pairs,
        "difference_acf_1": differences.autocorr(1) if len(differences) > 2 else np.nan,
        **stationarity_tests(regular),
    })

    if row.metric_id in plot_metrics and len(recording):
        step = max(1, len(recording) // MAX_PLOT_POINTS)
        shown = recording.iloc[::step]
        window = max(12, min(256, max(12, len(recording) // 20)))
        rolling_median = recording.rolling(window, min_periods=max(3, window // 3)).median().iloc[::step]
        rolling_q25 = recording.rolling(window, min_periods=max(3, window // 3)).quantile(0.25).iloc[::step]
        rolling_q75 = recording.rolling(window, min_periods=max(3, window // 3)).quantile(0.75).iloc[::step]
        fig, ax = plt.subplots(figsize=(12, 4))
        ax.plot(shown.index, shown, alpha=0.45, label="value")
        ax.plot(shown.index, rolling_median, color="black", label=f"rolling median ({window} observations)")
        ax.fill_between(shown.index, rolling_q25, rolling_q75, alpha=0.2, label="rolling IQR")
        ax.set_title(f"{row.metric_id} | {row.entity_id} | longest recording segment")
        ax.legend()
        save_and_show(fig, f"series_{row.metric_id}.png")

    if row.metric_id in plot_metrics and len(regular) >= 100 and regular.nunique() >= 3:
        values = regular.iloc[:10_000]
        lags = min(80, len(values) // 4)
        fig, axes = plt.subplots(1, 2, figsize=(13, 4))
        plot_acf(values, lags=lags, ax=axes[0], zero=False)
        plot_pacf(values, lags=lags, ax=axes[1], zero=False, method="ywm")
        axes[0].set_title(f"{row.metric_id}: ACF")
        axes[1].set_title(f"{row.metric_id}: PACF")
        save_and_show(fig, f"acf_pacf_{row.metric_id}.png")

temporal_evidence = pd.DataFrame(temporal_rows)
display(temporal_evidence)


## 9. Periodicity evidence and guarded STL

Period detection uses harmonic regression on the irregular observations, so
scattered gaps do not fragment the evidence. The reported strength is the
additional variance explained beyond a linear trend.

STL is only a visualization after short-gap filling leaves a sufficiently
long regular segment. Separate candidate decompositions are descriptive and
must not be interpreted as uniquely separating multiple seasonalities.


In [ ]:
PERIOD_SECONDS = {
    "four_hour": 4 * 60 * 60,
    "daily": 24 * 60 * 60,
    "weekly": 7 * 24 * 60 * 60,
}

def harmonic_strength(series, period_seconds, max_points=20_000):
    series = series.dropna()
    if len(series) > max_points:
        positions = np.linspace(0, len(series) - 1, max_points).astype(int)
        series = series.iloc[positions]
    elapsed = (series.index - series.index[0]).total_seconds().to_numpy()
    values = series.to_numpy(dtype=float)
    if len(values) < 100 or elapsed[-1] < MIN_STL_CYCLES * period_seconds:
        return np.nan, len(values), elapsed[-1] / period_seconds if len(values) else 0
    scaled_time = elapsed / elapsed[-1]
    base = np.column_stack([np.ones(len(values)), scaled_time])
    phase = 2 * np.pi * elapsed / period_seconds
    full = np.column_stack([base, np.sin(phase), np.cos(phase)])
    base_residual = values - base @ np.linalg.lstsq(base, values, rcond=None)[0]
    full_residual = values - full @ np.linalg.lstsq(full, values, rcond=None)[0]
    base_rss = np.sum(base_residual ** 2)
    strength = max(0.0, 1 - np.sum(full_residual ** 2) / base_rss) if base_rss > 0 else 0.0
    return strength, len(values), elapsed[-1] / period_seconds

def stl_strength(result):
    residual_variance = np.nanvar(result.resid)
    trend_denominator = np.nanvar(result.trend + result.resid)
    seasonal_denominator = np.nanvar(result.seasonal + result.resid)
    return {
        "trend_strength": max(0.0, 1 - residual_variance / trend_denominator) if trend_denominator > 0 else 0.0,
        "seasonal_strength": max(0.0, 1 - residual_variance / seasonal_denominator) if seasonal_denominator > 0 else 0.0,
    }

seasonality_rows = []
stl_results = {}
for row in representatives.itertuples(index=False):
    metadata = catalogue_index.loc[row.metric_id]
    series = raw_series(row.entity_id, row.metric_id)
    cadence_seconds = pd.to_numeric(metadata["expected_cadence_seconds"], errors="coerce")
    if pd.isna(cadence_seconds):
        regular = series.iloc[:0]
        fill_fraction, regular_missing_rate = np.nan, np.nan
    else:
        cadence = pd.Timedelta(seconds=float(cadence_seconds))
        regular, fill_fraction, regular_missing_rate = regular_segment(
            series, cadence, metadata["sampling_mode"]
        )

    for period_name, period_seconds in PERIOD_SECONDS.items():
        result_row = {
            "metric_id": row.metric_id,
            "entity_id": row.entity_id,
            "period_name": period_name,
            "period_seconds": period_seconds,
            "harmonic_strength": np.nan,
            "period_detection_n": 0,
            "cycles_available": 0,
            "period_detection_status": "not_applicable_sampling_mode",
            "decomposition_status": "not_attempted",
            "decomposition_n": len(regular),
            "short_gap_fill_fraction": fill_fraction,
            "regular_missing_rate": regular_missing_rate,
            "trend_strength": np.nan,
            "seasonal_strength": np.nan,
        }
        if metadata["sampling_mode"] == "periodic" and pd.notna(cadence_seconds):
            strength, n_used, cycles = harmonic_strength(series, period_seconds)
            result_row.update({
                "harmonic_strength": strength,
                "period_detection_n": n_used,
                "cycles_available": cycles,
                "period_detection_status": "evaluated" if pd.notna(strength) else "insufficient_span",
            })
            period_observations = round(period_seconds / float(cadence_seconds))
            if period_observations >= 2 and len(regular) >= MIN_STL_CYCLES * period_observations and regular.nunique() >= 3:
                try:
                    result = STL(regular.astype(float), period=period_observations, robust=True).fit()
                except (ValueError, np.linalg.LinAlgError):
                    result_row["decomposition_status"] = "fit_failed"
                else:
                    result_row.update({"decomposition_status": "evaluated", **stl_strength(result)})
                    stl_results[(row.metric_id, period_name)] = result
            else:
                result_row["decomposition_status"] = "insufficient_regular_segment"
        seasonality_rows.append(result_row)

seasonality_evidence = pd.DataFrame(seasonality_rows)
display(seasonality_evidence)

evaluated = seasonality_evidence.loc[
    seasonality_evidence["decomposition_status"].eq("evaluated")
].sort_values("seasonal_strength", ascending=False)
for row in evaluated.head(4).itertuples(index=False):
    result = stl_results[(row.metric_id, row.period_name)]
    figure = result.plot()
    figure.set_size_inches(12, 8)
    figure.suptitle(f"{row.metric_id}: robust STL ({row.period_name})", y=1.01)
    save_and_show(figure, f"stl_{row.metric_id}_{row.period_name}.png")


## 10. Cross-metric dependence

Correlation is calculated within each entity and summarized across entities.
Both levels and first differences are reported because shared trends can
create misleading level correlations.


In [ ]:
continuous_metrics = catalogue.loc[
    ~catalogue["measurement_kind"].eq("discrete_state"), "metric_id"
].head(12).tolist()
level_matrices, difference_matrices, count_matrices = [], [], []

for entity_id in selected_entities:
    entity = calibration.loc[
        calibration["entity_id"].astype(str).eq(entity_id)
        & calibration["metric_id"].isin(continuous_metrics),
        ["event_ts", "metric_id", "value"],
    ]
    pivot = entity.pivot_table(
        index="event_ts", columns="metric_id", values="value", aggfunc="first"
    ).reindex(columns=continuous_metrics).sort_index()
    if pivot.empty:
        continue
    level_matrices.append(pivot.corr(method="spearman").to_numpy())
    difference_matrices.append(pivot.diff().corr(method="spearman").to_numpy())
    count_matrices.append((pivot.notna().astype(int).T @ pivot.notna().astype(int)).to_numpy())

median_level = pd.DataFrame(
    np.nanmedian(np.stack(level_matrices), axis=0),
    index=continuous_metrics,
    columns=continuous_metrics,
)
median_difference = pd.DataFrame(
    np.nanmedian(np.stack(difference_matrices), axis=0),
    index=continuous_metrics,
    columns=continuous_metrics,
)
median_counts = pd.DataFrame(
    np.nanmedian(np.stack(count_matrices), axis=0),
    index=continuous_metrics,
    columns=continuous_metrics,
)

dependence_rows = []
for left_index, left in enumerate(continuous_metrics):
    for right in continuous_metrics[left_index + 1:]:
        dependence_rows.append({
            "metric_left": left,
            "metric_right": right,
            "median_level_spearman": median_level.loc[left, right],
            "median_difference_spearman": median_difference.loc[left, right],
            "median_pair_observations": median_counts.loc[left, right],
            "entities_assessed": len(level_matrices),
        })
dependence_evidence = pd.DataFrame(dependence_rows)
display(dependence_evidence.reindex(
    dependence_evidence["median_difference_spearman"].abs().sort_values(ascending=False).index
).head(20))

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
sns.heatmap(median_level, vmin=-1, vmax=1, center=0, cmap="vlag", ax=axes[0])
sns.heatmap(median_difference, vmin=-1, vmax=1, center=0, cmap="vlag", ax=axes[1])
axes[0].set_title("Median within-entity Spearman: levels")
axes[1].set_title("Median within-entity Spearman: differences")
save_and_show(fig, "correlation_levels_and_differences.png")


## 11. Data sufficiency summary

This status is mechanical and descriptive. It is not a domain judgement and
does not select a model representation. Population claims are suppressed
when fewer than ten entity series support a metric.


In [ ]:
def metric_status(row):
    if row["series_count"] < MIN_POPULATION_SERIES:
        return "contract_fixture_only"
    if row["valid_rate_full_p50"] < 0.80:
        return "limited_value_validity"
    if pd.notna(row["coverage_p50"]) and row["coverage_p50"] < 0.80:
        return "limited_coverage"
    if pd.notna(row["coverage_p10"]) and row["coverage_p10"] < 0.50:
        return "heterogeneous_coverage"
    if pd.isna(row["coverage_p50"]):
        return "descriptive_only_coverage_not_applicable"
    return "ready_for_baseline_research"

metric_profile["data_sufficiency_status"] = metric_profile.apply(metric_status, axis=1)
display(metric_profile[[
    "metric_id", "measurement_kind", "unit", "sampling_mode",
    "series_count", "valid_rate_full_p10", "valid_rate_full_p50",
    "valid_rate_full_p90", "coverage_p10", "coverage_p50", "coverage_p90",
    "clipped_rate_full_p10", "clipped_rate_full_p50", "clipped_rate_full_p90",
    "data_sufficiency_status",
]])


## 12. Save reproducible evidence

Outputs are compact evidence tables and figures, not a duplicate telemetry
dataset. The manifest pins the canonical hashes, analysis configuration,
selected entities and every permitted input path.


In [ ]:
read_files = [manifest_path, catalogue_path, registry_path, gaps_path, *telemetry_parts]
relative_reads = sorted(str(path.resolve().relative_to(CORE_RESOLVED)) for path in read_files)
assert all(path.resolve().is_relative_to(CORE_RESOLVED) for path in read_files)

output_tables = {
    "analysis_windows": analysis_windows,
    "full_series_profile": full_series,
    "series_profile": series_profile,
    "metric_profile": metric_profile,
    "temporal_evidence": temporal_evidence,
    "seasonality_evidence": seasonality_evidence,
    "dependence_evidence": dependence_evidence,
}
eda_manifest = {
    "eda_version": "0.2.0",
    "contract_version": CORE_VERSION,
    "sector": SECTOR,
    "spec_core": str(CORE_ROOT),
    "spec_core_manifest_sha256": sha256_file(manifest_path),
    "canonical_content_hashes": core_manifest["canonical_content_hashes"],
    "configuration": {
        "calibration_fraction": CALIBRATION_FRACTION,
        "requested_entity_limit": ENTITY_LIMIT,
        "actual_entity_count": len(selected_entities),
        "max_analysis_rows": MAX_ANALYSIS_ROWS,
        "min_population_series": MIN_POPULATION_SERIES,
        "max_short_gap_fill": MAX_SHORT_GAP_FILL,
    },
    "selected_entities": selected_entities,
    "files_read": relative_reads,
    "truth_guard_passed": True,
    "global_duplicate_keys": int(structural.loc[0, "duplicate_keys"]),
    "output_rows": {name: len(frame) for name, frame in output_tables.items()},
    "figures": sorted(path.name for path in FIGURE_CACHE.glob("*.png")),
}

if SAVE_OUTPUTS:
    with immutable_directory(EDA_ROOT) as output:
        for name, frame in output_tables.items():
            frame.to_parquet(output / f"{name}.parquet", index=False)
        shutil.copytree(FIGURE_CACHE, output / "figures")
        write_json(output / "eda_manifest.json", eda_manifest)
    print("Saved EDA evidence:", EDA_ROOT)
else:
    print("SAVE_EDA_OUTPUTS=False: results were not written")

display(pd.Series({
    "truth_guard_passed": True,
    "global_duplicate_keys": eda_manifest["global_duplicate_keys"],
    "metrics_profiled": len(metric_profile),
    "series_profiled": len(series_profile),
    "figures": len(eda_manifest["figures"]),
    "next_stage": "Notebook 03 evaluation harness",
}, name="result").to_frame())

connection.close()
figure_temp.cleanup()
